# **Phase 2: Predictive Modeling**

### **[1] Objectve:**
Predict the probability of customer churn using historical customer data, so that high-risk customers can be identified for retention efforts.

### **[2] Decide WHAT Features are allowed:**
A feature is only allowed if it is known *before* the customer churns.

- Features Allowed:
  - Time--------------------> tenure
  - Price--------------------> MonthlyCharges, TotalCharges
  - Commitment-----------> Contract
  - Payment behavior------> PaymentMethod, PaperlessBilling
  - Services-----------------> InternetService, Add-ons
  - Demographics (weak)--> SeniorCitizen, Partner, Dependents

- Features NOT Allowed:
  - Anything using churn label
  - Post-churn information
  - Heuristic LTVs

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# convert datatype of 'TotalCharges' into integer

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

In [5]:
# Look for all the "NaN" values in TotalCharges
print(df[df['TotalCharges'].isna()])

Empty DataFrame
Columns: [customerID, gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges, Churn]
Index: []

[0 rows x 21 columns]


In [6]:
# Tenure Transformation (Customer Age)
df['tenure_years'] = df['tenure'] / 12

In [7]:
# Price Intensity
df['charge_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

In [8]:
# Contract Commitment Score
df['contract_commitment'] = df['Contract'].map({
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
})

df = df.drop(columns=['Contract'])

In [9]:
# Auto-Pay Indicator
df['auto_pay'] = df['PaymentMethod'].map({
    'Electronic check': 0,
    'Mailed check': 0,
    'Bank transfer (automatic)': 1,
    'Credit card (automatic)': 1
})

df = df.drop(columns=['PaymentMethod'])

In [10]:
# Early Life Risk Indicator
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)

In [11]:
# Service Count
service_cols = [
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

df['service_count'] = df[service_cols].apply(lambda x: (x == 'Yes').sum(), axis=1)

In [12]:
# Fibre risk Indicator
df['is_fiber'] = (df['InternetService'] == 'Fiber optic').astype(int)

In [13]:
# Features Sanity Check

features_to_check = [
    'tenure_years',
    'charge_per_tenure',
    'contract_commitment',
    'auto_pay',
    'service_count',
    'is_new_customer'
]

df[features_to_check].describe()

,tenure_years,charge_per_tenure,contract_commitment,auto_pay,service_count,is_new_customer
count,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000
mean,2.701816,5.714882,0.688567,0.435580,2.038111,0.309300
std,2.045438,8.567435,0.832934,0.495868,1.847161,0.462238
min,0.083333,0.264384,0.000000,0.000000,0.000000,0.000000
25%,0.750000,1.250000,0.000000,0.000000,0.000000,0.000000
50%,2.416667,2.073598,0.000000,0.000000,2.000000,0.000000
75%,4.583333,5.884842,1.000000,1.000000,3.000000,1.000000
max,6.000000,51.225000,2.000000,1.000000,6.000000,1.000000


**Feature Validation Summary**

- A minimal feature set (7 variables) selected to balance predictive signal and interpretability.
- Features represent customer lifecycle, pricing pressure, contractual commitment, engagement depth, and known high-risk segments.
- Redundant categorical expansions were intentionally excluded to reduce noise and overfitting.

In [14]:
selected_features = [
    'tenure_years',
    'charge_per_tenure',
    'contract_commitment',
    'auto_pay',
    'service_count',
    'is_fiber',
    'is_new_customer'
]

X = df[selected_features]
y = (df['Churn'] == 'Yes').astype(int)

In [15]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    solver='liblinear'
)

model.fit(X, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [16]:
coef_table = (
    pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': model.coef_[0]
    })
    .sort_values(by='Coefficient', ascending=False)
)

coef_table

,Feature,Coefficient
5,is_fiber,1.158772
6,is_new_customer,0.107044
4,service_count,0.100211
1,charge_per_tenure,0.048467
0,tenure_years,-0.162628
3,auto_pay,-0.266524
2,contract_commitment,-1.035892


### **[8]Prioritization Table (SIZE x RISK x VALUE)**

In [22]:
# Create segments

df['priority_segment'] = np.select(
    [
        (df['is_fiber'] == 1) & (df['contract_commitment'] == 0),
        (df['is_fiber'] == 1) & (df['is_new_customer'] == 1),
        (df['is_fiber'] == 1) & (df['contract_commitment'] > 0),
        (df['is_fiber'] == 0) & (df['is_new_customer'] == 1),
        (df['is_fiber'] == 0) & (df['contract_commitment'] == 0) & (df['tenure_years'] > 1),
    ],
    [
        'Fiber + Month-to-Month',
        'Fiber + New',
        'Fiber + Contract',
        'New Non-Fiber',
        'Month-to-Month (Tenured)'
    ],
    default='Stable / Low Risk'
)

In [23]:
# Size x Risk x Value

priority_table = (
    df
    .groupby('priority_segment')
    .agg(
        Customers=('customerID', 'count'),
        ChurnRate=('Churn', lambda x: (x == 'Yes').mean()),
        AvgMonthlyRevenue=('MonthlyCharges', 'mean')
    )
    .reset_index()
)

In [24]:
# Priority Score

priority_table['PriorityScore'] = (
    priority_table['Customers'] *
    priority_table['ChurnRate'] *
    priority_table['AvgMonthlyRevenue']
)

priority_table = priority_table.sort_values(
    by='PriorityScore',
    ascending=False
)

priority_table

,priority_segment,Customers,ChurnRate,AvgMonthlyRevenue,PriorityScore
1,Fiber + Month-to-Month,2128,0.546053,87.021194,101118.626974
4,New Non-Fiber,1252,0.313099,37.001358,14504.532268
0,Fiber + Contract,961,0.138398,101.392300,13485.175858
3,Month-to-Month (Tenured),669,0.167414,46.651271,5224.942302
5,Stable / Low Risk,2015,0.033747,47.067593,3200.596328
2,Fiber + New,7,0.285714,95.042857,190.085714


In [25]:
# Top 10%

priority_table['PriorityPct'] = (
    priority_table['PriorityScore'] /
    priority_table['PriorityScore'].sum()
)

priority_table

,priority_segment,Customers,ChurnRate,AvgMonthlyRevenue,PriorityScore,PriorityPct
1,Fiber + Month-to-Month,2128,0.546053,87.021194,101118.626974,0.734212
4,New Non-Fiber,1252,0.313099,37.001358,14504.532268,0.105316
0,Fiber + Contract,961,0.138398,101.392300,13485.175858,0.097915
3,Month-to-Month (Tenured),669,0.167414,46.651271,5224.942302,0.037938
5,Stable / Low Risk,2015,0.033747,47.067593,3200.596328,0.023239
2,Fiber + New,7,0.285714,95.042857,190.085714,0.001380


### **Immediately prioritize retention actions for Fiber + Month-to-Month customers.**

>**Fiber + Month-to-Month**
- Customers: 2,128
- Churn Rate: 54.6% (catastrophic)
- Avg Monthly Revenue: ₹87
- Priority Contribution: 73.4% of total churn value

*This single segment explains ~¾ of the problem.*

>**All other segments**

Segment --> Priority % -->Reality check

New Non-Fiber --> 10.5% --> Worth watching, not urgent

Fiber + Contract --> 9.8% --> High value but already protected

Month-to-Month (Tenured) --> 3.8% --> Low churn leverage

Stable / Low Risk --> 2.3% --> Do nothing

Fiber + New --> 0.1% --> Statistically irrelevant

>**Recommended Action**

**Target only the top ~20–25% of Fiber + Month-to-Month customers, ranked by churn probability × revenue.**

Actions:
- Push 12-month contract upgrade with incentive
- Bundle auto-pay enrollment
- Avoid blanket discounts — precision only

***Retaining even 10–15% of this segment protects a disproportionate share of revenue, since this group alone contributes over 70% of churn risk value.***

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [45]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

log_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [46]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

# ROC-AUC
roc_auc_log = roc_auc_score(
    y_test,
    log_model.predict_proba(X_test)[:, 1]
)

# PR-AUC
precision, recall, _ = precision_recall_curve(
    y_test,
    log_model.predict_proba(X_test)[:, 1]
)
pr_auc_log = auc(recall, precision)

roc_auc_log, pr_auc_log

(0.8183963133640553, 0.6133755310449286)

In [47]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42
)

xgb_model.fit(X_train, y_train)

/opt/conda/envs/anaconda-ai-2025.12-py312/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Optional[float]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[str], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load

In [48]:
roc_auc_xgb = roc_auc_score(
    y_test,
    xgb_model.predict_proba(X_test)[:, 1]
)

precision, recall, _ = precision_recall_curve(
    y_test,
    xgb_model.predict_proba(X_test)[:, 1]
)
pr_auc_xgb = auc(recall, precision)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost'],
    'ROC_AUC': [roc_auc_log, roc_auc_xgb],
    'PR_AUC': [pr_auc_log, pr_auc_xgb]
})

comparison

,Model,ROC_AUC,PR_AUC
0,Logistic Regression,0.818396,0.613376
1,XGBoost,0.812037,0.593304


In [49]:
coef_table = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

coef_table

,Feature,Coefficient
5,is_fiber,1.193428
6,is_new_customer,0.341771
4,service_count,0.114887
1,charge_per_tenure,0.052025
0,tenure_years,-0.089057
3,auto_pay,-0.295407
2,contract_commitment,-1.085418


In [50]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance

,Feature,Importance
2,contract_commitment,0.595741
5,is_fiber,0.221560
1,charge_per_tenure,0.063838
3,auto_pay,0.040406
0,tenure_years,0.039457
4,service_count,0.030663
6,is_new_customer,0.008334


In [51]:
df['churn_probability'] = xgb_model.predict_proba(X)[:, 1]

In [52]:
# Target top 10% highest risk customers
threshold = df['churn_probability'].quantile(0.90)

df['target_flag'] = (df['churn_probability'] >= threshold).astype(int)

# Recall captured
recall_at_10 = (
    df[df['target_flag'] == 1]['Churn']
    .eq('Yes')
    .mean()
)

recall_at_10

np.float64(0.765625)

In [53]:
baseline_churn = (df['Churn'] == 'Yes').mean()

lift_10 = recall_at_10 / baseline_churn
lift_10

np.float64(2.88061797752809)

In [54]:
# Hypothetical assumptions
avg_monthly_revenue = df['MonthlyCharges'].mean()
retention_success_rate = 0.30  # 30% churn reduction
months_saved = 6

saved_revenue = (
    df[df['target_flag'] == 1].shape[0] *
    recall_at_10 *
    retention_success_rate *
    avg_monthly_revenue *
    months_saved
)

saved_revenue

np.float64(62867.22158703071)

### ***Targeting the top 10% highest-risk customers could capture ~3× more churners than random selection and potentially save ₹62867.22 revenue over 6 months.***